# 🏦 Phase 4 — Attijari Bank : Tableaux de bord & KPIs
### Visualisation complète des résultats pour la présentation au jury
---
**Modules :**
1. Chargement de toutes les données
2. KPIs globaux
3. Dashboard Phase 2 (NLP)
4. Dashboard Phase 3 (RPA + Anomalies)
5. Dashboard de synthèse finale


---
## 📦 CELLULE 1 — Installation

In [7]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install','matplotlib', 'pandas', 'numpy', 'seaborn', '--quiet'])
print('✅ Installation terminée — exécuter la Cellule 2')

CalledProcessError: Command '['C:\\Users\\Amal Brinsi\\AppData\\Local\\Programs\\Python\\Python310\\python.exe', '-m', 'pip', 'install', 'matplotlib', 'pandas', 'numpy', 'seaborn', '--quiet']' returned non-zero exit status 1.

---
## 🚀 CELLULE 2 — Phase 4 complète (KPIs + Dashboards)

In [4]:
import json
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor']   = '#f8f9fa'

print('✅ Librairies chargées')

# ============================================================
#  MODULE 1 — CHARGEMENT
# ============================================================
print('\n' + '='*55)
print('  MODULE 1 — Chargement des données')
print('='*55)

df = pd.read_csv('attijari_bank_logs_10000.csv', sep=';')
df['timestamp']        = pd.to_datetime(df['timestamp'])
df['success']          = df['success'].astype(str).str.strip().map({'True': True, 'False': False})
df['duree_action_sec'] = pd.to_numeric(df['duree_action_sec'], errors='coerce').fillna(0.0)
df['error_code']       = df['error_code'].fillna('NONE')
df['heure']            = df['timestamp'].dt.hour
df['jour_semaine']     = df['timestamp'].dt.dayofweek
df['mois']             = df['timestamp'].dt.month

with open('resultats_phase2.json', 'r', encoding='utf-8') as f:
    p2 = json.load(f)
repetitions  = p2.get('repetitions',  [])
erreurs      = p2.get('erreurs',      [])
goulots      = p2.get('goulots',      [])
opportunites = p2.get('opportunites', [])

with open('decisions_phase3.json', 'r', encoding='utf-8') as f:
    p3 = json.load(f)
decisions = p3.get('decisions', [])

print(f'  ✅ Logs CSV       : {len(df):,}')
print(f'  ✅ Répétitions    : {len(repetitions)}')
print(f'  ✅ Erreurs        : {len(erreurs)}')
print(f'  ✅ Goulots        : {len(goulots)}')
print(f'  ✅ Décisions RPA  : {len(decisions)}')

# ============================================================
#  MODULE 2 — KPIs GLOBAUX
# ============================================================
print('\n' + '='*55)
print('  MODULE 2 — KPIs Globaux')
print('='*55)

kpis = {
    'total_logs'      : len(df),
    'taux_echec_pct'  : round((~df['success']).mean()*100, 1),
    'taux_succes_pct' : round(df['success'].mean()*100, 1),
    'nb_utilisateurs' : df['user_id'].nunique(),
    'nb_agences'      : df['agence'].nunique(),
    'nb_processus'    : df['processus'].nunique(),
    'nb_applications' : df['application'].nunique(),
    'duree_moy_sec'   : round(df['duree_action_sec'].mean(), 2),
    'nb_repetitions'  : len(repetitions),
    'nb_erreurs'      : len(erreurs),
    'nb_goulots'      : len(goulots),
    'nb_opportunites' : len(opportunites),
    'nb_decisions_rpa': len(decisions),
    'gain_estime_min' : sum(d.get('gain_temps_min', 0) for d in decisions),
    'nb_robots'       : len(set(d['robot'] for d in decisions))
}

# ── Dashboard KPIs
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
fig.suptitle('🏦 Attijari Bank — KPIs Globaux', fontsize=18, fontweight='bold')
kpi_cards = [
    ('Logs Analysés',   f"{kpis['total_logs']:,}",       '#2980b9'),
    ('Taux de Succès',  f"{kpis['taux_succes_pct']}%",  '#27ae60'),
    ('Taux d\'Échec',  f"{kpis['taux_echec_pct']}%",   '#e74c3c'),
    ('Utilisateurs',    str(kpis['nb_utilisateurs']),    '#8e44ad'),
    ('Décisions RPA',   str(kpis['nb_decisions_rpa']),   '#e67e22'),
    ('Gain Estimé',     f"{kpis['gain_estime_min']} min",'#27ae60'),
    ('Robots Déployés', str(kpis['nb_robots']),          '#2980b9'),
    ('Opportunités',    str(kpis['nb_opportunites']),    '#c0392b'),
]
for ax, (titre, valeur, couleur) in zip(axes.flat, kpi_cards):
    ax.set_facecolor(couleur)
    ax.text(0.5, 0.62, valeur, transform=ax.transAxes,
            ha='center', va='center', fontsize=26, fontweight='bold', color='white')
    ax.text(0.5, 0.25, titre, transform=ax.transAxes,
            ha='center', va='center', fontsize=11, color='white', alpha=0.9)
    ax.set_xticks([]); ax.set_yticks([])
    for spine in ax.spines.values(): spine.set_visible(False)
plt.tight_layout()
plt.savefig('phase4_kpis_globaux.png', dpi=150, bbox_inches='tight')
plt.show()
print('  ✅ phase4_kpis_globaux.png')

# ============================================================
#  MODULE 3 — DASHBOARD PHASE 2
# ============================================================
print('\n' + '='*55)
print('  MODULE 3 — Dashboard Phase 2 (NLP)')
print('='*55)

fig = plt.figure(figsize=(20, 14))
fig.suptitle('📊 Dashboard Phase 2 — Analyse NLP des Logs', fontsize=16, fontweight='bold')
gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.4, wspace=0.35)

# Succès vs Échecs
ax1 = fig.add_subplot(gs[0, 0])
counts = df['success'].value_counts()
labels = ['Succès' if v else 'Échec' for v in counts.index]
colors = ['#27ae60' if v else '#e74c3c' for v in counts.index]
bars = ax1.bar(labels, counts.values, color=colors, edgecolor='white', width=0.5)
for bar, val in zip(bars, counts.values):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+50,
             str(val), ha='center', fontweight='bold', fontsize=10)
ax1.set_title('Succès vs Échecs', fontweight='bold')
ax1.set_ylabel('Nombre de logs')
ax1.grid(axis='y', alpha=0.3)

# Top 5 actions échouées
ax2 = fig.add_subplot(gs[0, 1])
df_failed = df[~df['success']]
top_actions = df_failed['action'].value_counts().head(5)
ax2.barh(top_actions.index[::-1], top_actions.values[::-1], color='#e74c3c', edgecolor='white')
ax2.set_title('Top 5 Actions Échouées', fontweight='bold')
ax2.set_xlabel('Nombre d\'échecs')
ax2.grid(axis='x', alpha=0.3)

# Top 5 processus échoués
ax3 = fig.add_subplot(gs[0, 2])
top_proc = df_failed['processus'].value_counts().head(5)
ax3.barh(top_proc.index[::-1], top_proc.values[::-1], color='#e67e22', edgecolor='white')
ax3.set_title('Top 5 Processus Échoués', fontweight='bold')
ax3.set_xlabel('Nombre d\'échecs')
ax3.grid(axis='x', alpha=0.3)

# Taux d'échec par agence
ax4 = fig.add_subplot(gs[1, 0])
taux_agence = df.groupby('agence')['success'].apply(lambda x: (~x).mean()*100).sort_values(ascending=False)
colors_agence = ['#e74c3c' if v > 35 else '#e67e22' if v > 25 else '#27ae60' for v in taux_agence.values]
ax4.bar(range(len(taux_agence)), taux_agence.values, color=colors_agence, edgecolor='white')
ax4.set_xticks(range(len(taux_agence)))
ax4.set_xticklabels(taux_agence.index, rotation=45, ha='right', fontsize=7)
ax4.set_title('Taux d\'Échec par Agence (%)', fontweight='bold')
ax4.set_ylabel('%')
ax4.grid(axis='y', alpha=0.3)

# Activité par heure
ax5 = fig.add_subplot(gs[1, 1])
logs_heure   = df.groupby('heure').size()
echecs_heure = df[~df['success']].groupby('heure').size()
ax5.plot(logs_heure.index, logs_heure.values, color='#2980b9', linewidth=2, label='Total')
ax5.plot(echecs_heure.index, echecs_heure.values, color='#e74c3c', linewidth=2, label='Échecs')
ax5.fill_between(echecs_heure.index, echecs_heure.values, alpha=0.2, color='#e74c3c')
ax5.set_title('Activité par Heure', fontweight='bold')
ax5.set_xlabel('Heure')
ax5.set_ylabel('Logs')
ax5.legend()
ax5.grid(alpha=0.3)

# Top codes erreur pie
ax6 = fig.add_subplot(gs[1, 2])
top_err = df[df['error_code']!='NONE']['error_code'].value_counts().head(5)
ax6.pie(top_err.values, labels=top_err.index, autopct='%1.1f%%',
        colors=['#e74c3c','#e67e22','#f39c12','#c0392b','#8e44ad'],
        startangle=90, textprops={'fontsize': 8})
ax6.set_title('Top 5 Codes Erreur', fontweight='bold')

# Durée par application
ax7 = fig.add_subplot(gs[2, 0])
duree_app = df.groupby('application')['duree_action_sec'].mean().sort_values(ascending=False)
ax7.bar(duree_app.index, duree_app.values, color='#8e44ad', edgecolor='white')
ax7.set_title('Durée Moy. par Application (s)', fontweight='bold')
ax7.set_ylabel('Secondes')
ax7.set_xticklabels(duree_app.index, rotation=20, ha='right')
ax7.grid(axis='y', alpha=0.3)

# Répétitions
ax8 = fig.add_subplot(gs[2, 1])
rep_types = pd.Series([r['type'] for r in repetitions]).value_counts()
short_labels = [t.replace('REPETITION_','') for t in rep_types.index]
ax8.bar(short_labels, rep_types.values, color=['#3498db','#e74c3c','#27ae60'], edgecolor='white')
ax8.set_title('Types de Répétitions', fontweight='bold')
ax8.set_ylabel('Nombre')
ax8.set_xticklabels(short_labels, rotation=15, ha='right')
ax8.grid(axis='y', alpha=0.3)

# Logs par jour de semaine
ax9 = fig.add_subplot(gs[2, 2])
jours = ['Lun','Mar','Mer','Jeu','Ven','Sam','Dim']
logs_jour = df.groupby('jour_semaine').size()
ax9.bar([jours[i] for i in logs_jour.index], logs_jour.values, color='#2980b9', edgecolor='white')
ax9.set_title('Logs par Jour de Semaine', fontweight='bold')
ax9.set_ylabel('Nombre de logs')
ax9.grid(axis='y', alpha=0.3)

plt.savefig('phase4_dashboard_phase2.png', dpi=150, bbox_inches='tight')
plt.show()
print('  ✅ phase4_dashboard_phase2.png')

# ============================================================
#  MODULE 4 — DASHBOARD PHASE 3
# ============================================================
print('\n' + '='*55)
print('  MODULE 4 — Dashboard Phase 3 (RPA + Anomalies)')
print('='*55)

df_dec = pd.DataFrame(decisions)

fig = plt.figure(figsize=(20, 12))
fig.suptitle('🤖 Dashboard Phase 3 — Moteur de Décision RPA + Anomalies', fontsize=16, fontweight='bold')
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

# Décisions par type
ax1 = fig.add_subplot(gs[0, 0])
type_counts = df_dec['type_robot'].value_counts()
colors_type = ['#3498db','#e74c3c','#8e44ad','#e67e22']
bars = ax1.bar(type_counts.index, type_counts.values, color=colors_type[:len(type_counts)], edgecolor='white')
for bar, val in zip(bars, type_counts.values):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
             str(val), ha='center', fontweight='bold', fontsize=10)
ax1.set_title('Décisions par Type de Robot', fontweight='bold')
ax1.set_ylabel('Nombre')
ax1.grid(axis='y', alpha=0.3)

# Décisions par priorité
ax2 = fig.add_subplot(gs[0, 1])
prio_counts = df_dec['priorite'].value_counts().sort_index()
labels_prio = [f'Priorité {p}' for p in prio_counts.index]
colors_prio = ['#e74c3c','#e67e22','#3498db']
bars = ax2.bar(labels_prio, prio_counts.values, color=colors_prio[:len(prio_counts)], edgecolor='white')
for bar, val in zip(bars, prio_counts.values):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
             str(val), ha='center', fontweight='bold', fontsize=10)
ax2.set_title('Décisions par Priorité', fontweight='bold')
ax2.set_ylabel('Nombre')
ax2.grid(axis='y', alpha=0.3)

# Gain par type
ax3 = fig.add_subplot(gs[0, 2])
gain_type = df_dec.groupby('type_robot')['gain_temps_min'].sum().sort_values(ascending=False)
bars = ax3.bar(gain_type.index, gain_type.values,
               color=['#27ae60','#e74c3c','#3498db','#8e44ad'], edgecolor='white')
for bar, val in zip(bars, gain_type.values):
    ax3.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
             f'{val}m', ha='center', fontweight='bold', fontsize=9)
ax3.set_title('Gain Estimé par Type (min)', fontweight='bold')
ax3.set_ylabel('Minutes')
ax3.grid(axis='y', alpha=0.3)

# Robots les plus sollicités
ax4 = fig.add_subplot(gs[1, 0])
robot_counts = df_dec['robot'].value_counts()
ax4.barh(robot_counts.index[::-1], robot_counts.values[::-1], color='#2980b9', edgecolor='white')
for i, val in enumerate(robot_counts.values[::-1]):
    ax4.text(val+0.5, i, str(val), va='center', fontweight='bold', fontsize=9)
ax4.set_title('Robots les Plus Sollicités', fontweight='bold')
ax4.set_xlabel('Interventions')
ax4.grid(axis='x', alpha=0.3)

# Anomalies R008
ax5 = fig.add_subplot(gs[1, 1])
anomalies_r8 = df_dec[df_dec['regle_id']=='R008'].copy()
if len(anomalies_r8) > 0 and 'taux_echec_pct' in anomalies_r8.columns:
    anomalies_r8 = anomalies_r8.sort_values('taux_echec_pct', ascending=False).head(8)
    colors_anom = ['#e74c3c' if t >= 70 else '#e67e22' for t in anomalies_r8['taux_echec_pct']]
    ax5.bar(anomalies_r8['user_id'], anomalies_r8['taux_echec_pct'],
            color=colors_anom, edgecolor='white')
    ax5.axhline(y=50, color='red', linestyle='--', alpha=0.7, label='Seuil 50%')
    ax5.set_title('Anomalies R008 — Taux Échec Utilisateurs', fontweight='bold')
    ax5.set_ylabel('%')
    ax5.set_xticklabels(anomalies_r8['user_id'], rotation=30, ha='right', fontsize=8)
    ax5.legend(fontsize=8)
    ax5.grid(axis='y', alpha=0.3)
else:
    ax5.text(0.5, 0.5, 'Pas d\'anomalies R008', ha='center', va='center',
             transform=ax5.transAxes, fontsize=12)
    ax5.set_title('Anomalies R008', fontweight='bold')

# Sources des décisions
ax6 = fig.add_subplot(gs[1, 2])
source_counts = df_dec['source'].value_counts()
colors_src = ['#3498db','#e74c3c','#27ae60','#8e44ad']
ax6.pie(source_counts.values, labels=source_counts.index,
        autopct='%1.1f%%', colors=colors_src[:len(source_counts)],
        startangle=90, textprops={'fontsize': 9})
ax6.set_title('Sources des Décisions RPA', fontweight='bold')

plt.savefig('phase4_dashboard_phase3.png', dpi=150, bbox_inches='tight')
plt.show()
print('  ✅ phase4_dashboard_phase3.png')

# ============================================================
#  MODULE 5 — DASHBOARD SYNTHÈSE FINALE
# ============================================================
print('\n' + '='*55)
print('  MODULE 5 — Synthèse Finale')
print('='*55)

fig = plt.figure(figsize=(20, 10))
fig.suptitle('🏆 Synthèse Finale — Système d\'Amélioration Continue Attijari Bank',
             fontsize=16, fontweight='bold')
gs = gridspec.GridSpec(2, 4, figure=fig, hspace=0.45, wspace=0.35)

# Pipeline
ax1 = fig.add_subplot(gs[0, 0:2])
phases  = ['Phase 1\nCollecte', 'Phase 2\nNLP', 'Phase 3\nRPA', 'Phase 4\nKibana']
valeurs = [10000, len(repetitions)+len(erreurs)+len(goulots), len(decisions), len(decisions)]
colors_phases = ['#3498db','#27ae60','#e67e22','#8e44ad']
bars = ax1.bar(phases, valeurs, color=colors_phases, edgecolor='white', width=0.5)
for bar, val in zip(bars, valeurs):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+10,
             str(val), ha='center', fontweight='bold', fontsize=11)
ax1.set_title('Pipeline Complet — Données traitées par Phase', fontweight='bold')
ax1.set_ylabel('Nombre d\'éléments')
ax1.grid(axis='y', alpha=0.3)

# Taux succès pie
ax2 = fig.add_subplot(gs[0, 2])
ax2.pie([kpis['taux_succes_pct'], kpis['taux_echec_pct']],
        labels=[f'Succès\n{kpis["taux_succes_pct"]}%', f'Échecs\n{kpis["taux_echec_pct"]}%'],
        colors=['#27ae60','#e74c3c'], autopct='%1.1f%%',
        startangle=90, textprops={'fontsize': 10})
ax2.set_title('Taux de Succès Global', fontweight='bold')

# Top agences
ax3 = fig.add_subplot(gs[0, 3])
echecs_agence = df[~df['success']].groupby('agence').size().sort_values(ascending=False).head(5)
ax3.barh(echecs_agence.index[::-1], echecs_agence.values[::-1], color='#e74c3c', edgecolor='white')
ax3.set_title('Top 5 Agences — Échecs', fontweight='bold')
ax3.set_xlabel('Nombre d\'échecs')
ax3.grid(axis='x', alpha=0.3)

# Évolution mensuelle
ax4 = fig.add_subplot(gs[1, 0:2])
mois_labels = ['Jan','Fév','Mar','Avr','Mai','Jun','Jul','Aoû','Sep','Oct','Nov','Déc']
logs_mois   = df.groupby('mois').size()
echecs_mois = df[~df['success']].groupby('mois').size()
ax4.fill_between(logs_mois.index, logs_mois.values, alpha=0.3, color='#2980b9')
ax4.fill_between(echecs_mois.index, echecs_mois.values, alpha=0.4, color='#e74c3c')
ax4.plot(logs_mois.index, logs_mois.values, color='#2980b9', linewidth=2, label='Total logs')
ax4.plot(echecs_mois.index, echecs_mois.values, color='#e74c3c', linewidth=2, label='Échecs')
ax4.set_xticks(logs_mois.index)
ax4.set_xticklabels([mois_labels[m-1] for m in logs_mois.index])
ax4.set_title('Évolution Mensuelle des Logs', fontweight='bold')
ax4.set_ylabel('Nombre de logs')
ax4.legend()
ax4.grid(alpha=0.3)

# Répartition RPA
ax5 = fig.add_subplot(gs[1, 2])
rpa_types  = ['Reconnexion','Correction','Optimisation','Anomalie']
rpa_counts = [
    len([d for d in decisions if d['type_robot']=='RECONNEXION']),
    len([d for d in decisions if d['type_robot']=='CORRECTION']),
    len([d for d in decisions if d['type_robot']=='OPTIMISATION']),
    len([d for d in decisions if d['type_robot']=='ANOMALIE'])
]
colors_rpa = ['#3498db','#e74c3c','#8e44ad','#e67e22']
ax5.pie([c for c in rpa_counts if c > 0],
        labels=[rpa_types[i] for i,c in enumerate(rpa_counts) if c > 0],
        colors=[colors_rpa[i] for i,c in enumerate(rpa_counts) if c > 0],
        autopct='%1.1f%%', startangle=90, textprops={'fontsize': 9})
ax5.set_title('Répartition Robots RPA', fontweight='bold')

# Gain total
gain_total = kpis['gain_estime_min']
ax6 = fig.add_subplot(gs[1, 3])
ax6.set_facecolor('#27ae60')
ax6.text(0.5, 0.60, f'{gain_total}', transform=ax6.transAxes,
         ha='center', va='center', fontsize=36, fontweight='bold', color='white')
ax6.text(0.5, 0.30, 'minutes économisées', transform=ax6.transAxes,
         ha='center', va='center', fontsize=11, color='white')
ax6.text(0.5, 0.10, f'soit {gain_total/60:.1f} heures', transform=ax6.transAxes,
         ha='center', va='center', fontsize=10, color='white', alpha=0.8)
ax6.set_xticks([]); ax6.set_yticks([])
for spine in ax6.spines.values(): spine.set_visible(False)
ax6.set_title('💰 Gain Estimé Total', fontweight='bold')

plt.savefig('phase4_synthese_finale.png', dpi=150, bbox_inches='tight')
plt.show()
print('  ✅ phase4_synthese_finale.png')

# ============================================================
#  RAPPORT FINAL
# ============================================================
print(f"""
{'='*55}
  RAPPORT FINAL — Phase 4 : Tableaux de bord & KPIs
  Attijari Bank
{'='*55}

📊 DONNÉES ANALYSÉES
   Logs totaux          : {kpis['total_logs']:,}
   Période              : {df['timestamp'].min().date()} → {df['timestamp'].max().date()}
   Agences              : {kpis['nb_agences']}
   Utilisateurs         : {kpis['nb_utilisateurs']}
   Applications         : {kpis['nb_applications']}
   Processus            : {kpis['nb_processus']}

✅ PERFORMANCE
   Taux de succès       : {kpis['taux_succes_pct']}%
   Taux d'échec         : {kpis['taux_echec_pct']}%
   Durée moy. action    : {kpis['duree_moy_sec']}s

🔍 PROBLÈMES DÉTECTÉS (Phase 2)
   Répétitions          : {kpis['nb_repetitions']}
   Erreurs fréquentes   : {kpis['nb_erreurs']}
   Goulots              : {kpis['nb_goulots']}
   Opportunités         : {kpis['nb_opportunites']}

🤖 ROBOTS RPA (Phase 3)
   Total décisions      : {kpis['nb_decisions_rpa']}
   Robots déployés      : {kpis['nb_robots']}
   Gain estimé          : {kpis['gain_estime_min']} minutes
   Soit                 : {kpis['gain_estime_min']/60:.1f} heures

📁 FICHIERS GÉNÉRÉS
   phase4_kpis_globaux.png
   phase4_dashboard_phase2.png
   phase4_dashboard_phase3.png
   phase4_synthese_finale.png

✅ Phase 4 terminée !
   → Phase 5 : Tests, rapport final, diagrammes UML
{'='*55}""")

ModuleNotFoundError: No module named 'seaborn'